In [1]:
import torch

from rl.maddpg import *
from custom_envs.diff_driven.gym_env.centered_paralelenv import *


In [2]:
seed=41
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
env=DiffDriveParallelEnvDone(v_ang_max=torch.pi/12, num_obstacles=10)

In [4]:
env.reset()

{'agent_0': array([-0.02759166,  0.20626095,  0.31438687, -0.02225225,  0.5081886 ,
        -0.2724051 ,  0.504587  ,  0.31348783,  0.48444504,  0.43222165,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        -0.4424796 , -0.6189331 , -0.41774526,  0.99826807, -0.7618759 ,
         0.8967786 , -0.7854437 ,  0.9085642 ,  0.05882937, -0.647723  ,
         0.31856766, -0.13414472,  0.3573635 ,  0.09790049,  0.41628855,
         0.22108808,  0.4891513 , -0.23500076,  0.676529  , -0.1466417 ,
         0.6906829 ,  0.0727381 ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.165659  ,  0.60840905,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        , -0.4182524 ,  0.59189886,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.90833086,  0.8060123 ,
         0.        ,  0.        ,  0.   

In [5]:
n=6
spacing=3.0

In [6]:
import torch
import math

# def create_symmetric_tree_pattern():
#     R = 15 / 2  # Max radius to fit inside 15x15 square
#
#     positions = []
#
#     # Top point (root)
#     positions.append(torch.tensor([0.0, R]))
#
#     # Level 1 (2 points at 120° and 240° from top)
#     for angle_deg in [210, 330]:
#         angle_rad = math.radians(angle_deg)
#         x = R * math.cos(angle_rad)
#         y = R * math.sin(angle_rad)
#         positions.append(torch.tensor([x, y]))
#
#     # Level 2 (3 bottom points at 0°, 120°, 240°)
#     for angle_deg in [270, 30, 150]:
#         angle_rad = math.radians(angle_deg)
#         x = R * math.cos(angle_rad)
#         y = R * math.sin(angle_rad)
#         positions.append(torch.tensor([x, y]))
#
#     return torch.stack(positions)

In [7]:
# def triangle_tree_layout():
#     # Define the vertical spacing
#     h = 30 / 2  # Half height of square
#     dy = h / 2  # Vertical distance between levels
#
#     # Level 0 (top)
#     top = torch.tensor([0.0, dy])
#
#     # Level 1 (2 points)
#     mid_left = torch.tensor([-dy / math.sqrt(3), 0.0])
#     mid_right = torch.tensor([dy / math.sqrt(3), 0.0])
#
#     # Level 2 (3 points)
#     bottom_y = -dy
#     bottom_radius = 2*dy / math.sqrt(3)
#     bottom_left = torch.tensor([-bottom_radius, bottom_y])
#     bottom_center = torch.tensor([0.0, bottom_y])
#     bottom_right = torch.tensor([bottom_radius, bottom_y])
#
#     return torch.stack([top, mid_left, mid_right, bottom_left, bottom_center, bottom_right])

In [8]:
# x_positions = torch.linspace(-(n - 1) / 2 * spacing, (n - 1) / 2 * spacing, n)

In [9]:
# radius=5

In [10]:

# angles = torch.arange(n) * (2 * torch.pi / n)
# x = radius * torch.cos(angles)
# y = radius * torch.sin(angles)


In [11]:
# env.landmarks=torch.stack((x, y), dim=1).to(device=device)#torch.stack(( torch.zeros(n), x_positions), dim=1).to(device=device)


In [12]:
# env.agent_pos=triangle_tree_layout().to(device=device)#torch.stack((x, y), dim=1).to(device=device)

In [13]:
# env.obstacle_pos+=20

In [14]:
# env._reset_hungarian()

In [15]:
# env.agent_dir=torch.zeros(6, device=device)

In [16]:
actor_weight_path='C:\\Users\\abzza\\Desktop\\run\\done350\\episode_500__shared_actor.pth'

In [17]:
maddpg=MADDPGSharedActorCriticIndependentQmean(env, reward_scales=[
            1.0,      # progressive
            1.0,      # distance
            0.0,      # base
            1000,   # reached goal
            200.0,    # agent collision
            200.0,    # obstacle collision
            0,      # v_lin
            0.,      # v_ang
            1       # time
        ])

In [18]:
maddpg.actor.load_checkpoint(actor_weight_path)

Checkpoint loaded from C:\Users\abzza\Desktop\run\done350\episode_500__shared_actor.pth


In [19]:
# trajectory, _ = maddpg.try_actor(reset=False)


In [20]:
# maddpg.plot_episode_gone_trajectory(np.stack(trajectory), 'test')

In [ ]:
import numpy as np
import torch

success_counter = 0
episode_idx = 0


while success_counter < 1000:
    env.reset()
    with open("log.txt", "a") as f:
        print(f"\n--- Episode {episode_idx} ---", file=f)
        print(f"Initial positions: {env.agent_pos.cpu().numpy()}", file=f)
        print(f"Initial headings: {env.agent_dir}", file=f)

    # Run policy and record trajectory
    trajectory, score, time_to_completion, distance_per_agent  = maddpg.try_actor(reset=True)
    tagged_count = env.dones.sum().item()

    if tagged_count == 6:
        success_counter += 1
        # Save trajectory plot
        maddpg.plot_episode_gone_trajectory(np.stack(trajectory), f"test{episode_idx}_tagged_{tagged_count}")

        # Log extra metrics
        with open("log.txt", "a") as f:
            print(f"Episode {episode_idx} score: {score}, tagged: {tagged_count}", file=f)
            print(f"Time to completion per agent: {time_to_completion}", file=f)
            print(f"Distance traveled per agent: {distance_per_agent}", file=f)

    else:
        with open("log.txt", "a") as f:
            print(f"Episode {episode_idx} score: {score}, tagged: {tagged_count} (not saved)", file=f)
            print(f"Time to completion per agent: {time_to_completion}", file=f)
            print(f"Distance traveled per agent: {distance_per_agent}", file=f)

    print(f"Episode {episode_idx} score: {score} tagged: {tagged_count}")
    episode_idx += 1


trying current actor
100 steps passed
200 steps passed
300 steps passed
400 steps passed
500 steps passed
score: tensor([[ 4.3626e+00, -2.2901e+00, -4.1688e+00,  1.0000e+00, -1.8238e-03,
          0.0000e+00, -8.0000e+00, -1.0747e+01, -1.8000e+01],
        [-3.4819e-01, -8.0848e+00, -8.4364e+00,  1.0000e+00,  0.0000e+00,
         -6.7498e-03, -2.1000e+01, -4.9268e+01, -4.6000e+01],
        [ 7.8404e+01, -1.7691e+02, -5.9365e+01,  0.0000e+00,  0.0000e+00,
         -7.2951e-03, -6.7965e+01, -3.7343e+02, -5.0000e+02],
        [ 5.6965e+00, -2.6322e+01, -1.5768e+01,  1.0000e+00, -9.1188e-04,
         -1.6923e-03, -3.1000e+01, -9.5066e+01, -1.0100e+02],
        [ 4.6162e+00, -7.2438e-01, -1.4831e+00,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00, -2.4687e+00, -6.0000e+00],
        [-8.1471e+01, -1.4186e+02, -5.9365e+01,  0.0000e+00, -2.7356e-03,
         -1.4590e-02, -4.2002e+01, -6.0500e+02, -5.0000e+02]], device='cuda:0'), done: 4
Episode 0 score: tensor([[ 4.3626e+00, -2.29

In [ ]:
success_counter=0
i=0
while success_counter<100:
    env.reset()
    with open('log.txt', 'a') as f:
        print(f'Episode:{i}', file=f)
        print(f'Initial positions:{env.agent_pos}', file=f)
        print(f'Initial headings: {env.agent_dir}', file=f)
    # env.obstacle_pos+=50
    trajectory,score = maddpg.try_actor(reset=True)
    tagged_count=env.dones.sum().item()
    if tagged_count==6:
        success_counter+=1
        maddpg.plot_episode_gone_trajectory(np.stack(trajectory), f'test{i}_tagged_{tagged_count}')
    print(f'Episode {i} score: {score} tagged:{tagged_count} ')
    with open('log.txt', 'a') as f:
        print(f'Episode {i} score: {score} tagged:{tagged_count} ', file=f)

    i=i+1



trying current actor
100 steps passed
score: tensor([  0.0000,   0.0000, 999.5279,   0.0000,   0.0000,   0.0000],
       device='cuda:0'), done: 6
Episode 0 score: tensor([  0.0000,   0.0000, 999.5279,   0.0000,   0.0000,   0.0000],
       device='cuda:0') tagged:6 
trying current actor
100 steps passed
200 steps passed
300 steps passed
400 steps passed
500 steps passed
score: tensor([ 0.0000, -0.7135,  0.0000, -0.2189, -0.8705,  0.0000], device='cuda:0'), done: 3
Episode 1 score: tensor([ 0.0000, -0.7135,  0.0000, -0.2189, -0.8705,  0.0000], device='cuda:0') tagged:3 
trying current actor
100 steps passed
200 steps passed
300 steps passed
400 steps passed
500 steps passed
score: tensor([-1.4929,  0.0000, -0.5616,  0.0000,  0.0000,  0.0000], device='cuda:0'), done: 4
Episode 2 score: tensor([-1.4929,  0.0000, -0.5616,  0.0000,  0.0000,  0.0000], device='cuda:0') tagged:4 
trying current actor
100 steps passed
200 steps passed
300 steps passed
400 steps passed
500 steps passed
score: te